# Reconnaissance d'entités nommées avec SpaCy

La documentation est accessible ici: https://spacy.io/api

## Reconnaissance d'entités nommées : année 1964

**Objectif** : trouver les personnes, les organisations et les lieux les plus présents dans les journaux de 1964.

**Méthode** : spaCy repère les entités dans chaque journal. Pour chaque entité, on compte le nombre de journaux où elle apparaît, et on garde les 20 plus présentes.

**Plan en 5 étapes** :

1. Préparer les journaux de 1964
2. Les personnes (PER)
3. Les organisations (ORG)
4. Les lieux (LOC)
5. Conclusion

## Étape 1 : préparer les journaux de 1964


In [2]:
import spacy

# Charger le modèle français : il contient déjà la reconnaissance des entités
nlp = spacy.load("fr_core_news_md")

/home/minasema/tac/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:1174: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [3]:
# Test sur une phrase dont on connaît les entités
doc = nlp("Pierre Harmel est arrivé à Bruxelles pour le gouvernement belge.")

In [4]:
# Afficher chaque entité trouvée et son étiquette
for ent in doc.ents:
    print(ent.text, ent.label_)

Pierre Harmel PER
Bruxelles LOC


In [5]:
from collections import Counter
import os

# Depuis tps/tp3, le dossier des journaux est deux niveaux plus haut
txt_path = "../../data/txt"
year = 1964

# Contrôle : le dossier existe-t-il ?
print(os.path.isdir(txt_path))

True


In [6]:
# Liste des fichiers dont le nom contient 1964
txts = [f for f in os.listdir(txt_path) if os.path.isfile(os.path.join(txt_path, f)) and str(year) in f]

In [7]:
print(len(txts), "journaux de", year)

100 journaux de 1964


## Étape 2 : les personnes

Nous cherchons les entités de type **PER**. Pour chaque journal, une personne n'est comptée qu'une fois. Nous affichons ensuite la liste des 20 personnes présentes dans le plus de journaux.

Avant de compter, nous regardons les entités d'un seul journal, pour voir exactement ce que spaCy a repéré.

In [8]:
# Lire un journal et afficher les 30 premières entités avec leur étiquette
with open(os.path.join(txt_path, txts[0]), "r", encoding="utf-8") as f:
    doc = nlp(f.read())

for ent in doc.ents[:30]:
    print(repr(ent.text), ent.label_)

'VENDREDI 3' MISC
'Porter' PER
'Bon Marché' MISC
'Loterie Nationale' ORG
"Tirage du-6 avril'" MISC
'Blouse' LOC
'Vichy' LOC
'ÔO Fr' ORG
'marine mauve' PER
'Fr + cac j eau ® dixièmes 2 Blouse blanche' ORG
'Fr + Ga' ORG
'NON ILLUSTRE' MISC
'Jupe droite blanche' MISC
'Dior 350 Fr + Ga' MISC
'SAvn' LOC
'Blouse' LOC
'Fr' LOC
'Jupe droite en toile de fibranne' MISC
'Fr' LOC
'B' LOC
'I' LOC
'Jlgslc' MISC
'van' PER
'Blouse' LOC
'Fr' LOC
'Gac' PER
'Jupe droite' MISC
'Raylon' PER
'Fr + ca deau' MISC
'UX' LOC


In [9]:
presence_per = Counter()

for txt in txts:
    with open(os.path.join(txt_path, txt), "r", encoding="utf-8") as f:
        doc = nlp(f.read())

    # set() : chaque personne n'est comptée qu'une fois par journal
    personnes = {ent.text for ent in doc.ents if ent.label_ == "PER" and len(ent.text) > 4}
    presence_per.update(personnes)

print(len(presence_per), "personnes différentes")

13692 personnes différentes


In [10]:
for rang, (nom, n) in enumerate(presence_per.most_common(200), start=1):
    print(f"{rang:2}. {nom} : {n} journaux")

 1. Rossel : 30 journaux
 2. Louise : 27 journaux
 3. Ecrire Ag : 21 journaux
 4. - Tél : 21 journaux
 5. A. P. : 20 journaux
 6. A. F. P. : 16 journaux
 7. Albert : 15 journaux
 8. Reine : 14 journaux
 9. F. Tél : 13 journaux
10. Johnson : 13 journaux
11. Bonne : 12 journaux
12. Martin : 12 journaux
13. F. T. : 12 journaux
14. Molière : 11 journaux
15. A.F.P. : 11 journaux
16. Mozart : 11 journaux
17. Brugmann : 10 journaux
18. Beethoven : 10 journaux
19. Janssens : 10 journaux
20. Président : 9 journaux
21. Shakespeare : 9 journaux
22. Ag Rossel : 9 journaux
23. Couronne : 9 journaux
24. Belliard : 8 journaux
25. Louis XV : 8 journaux
26. Rosse : 8 journaux
27. Léopold : 7 journaux
28. prince Albert : 7 journaux
29. M. Johnson : 7 journaux
30. Herman : 7 journaux
31. Ec Ag : 7 journaux
32. s.d.b : 7 journaux
33. Monsieur : 7 journaux
34. Bruxel : 7 journaux
35. b. ét : 7 journaux
36. R. T. F. : 7 journaux
37. R. T. B. : 7 journaux
38. Madou : 7 journaux
39. O. N. U. : 7 journaux
40. 

## Étape 2b : nettoyer la liste des personnes

La liste brute contient des entrées qui ne sont pas des personnes. Nous les avons repérées dans la liste de l'étape 2a, et nous notons la raison de chaque retrait :

| Entrée | Raison |
|---|---|
| ROSSEL, AGENCE ROSSEL | groupe de presse : une organisation |
| ECRIRE AG, TÉL, F. TÉL, EC AG | abréviations des petites annonces |
| BONNE, LUNDI, FAIRE, TEMPS, JARDIN, OFFRE, LIBRE, TEINT | mots ordinaires, pris pour des noms propres à cause des majuscules |
| BRUGMANN, LOUISE, MADOU, ANSPACH | probablement des lieux (hôpital, avenue, place, boulevard) |
| PRÉSIDENT, MONSIEUR, MADAME | titres sans nom |
| A. P., A. F. P., A.F.P. | sigles (avec un point) : agences de presse, à retrouver à l'étape 3 |

Deux corrections sont aussi faites : on enlève le tiret du début (« - Tél »), et on met tout en majuscules, pour que « Monsieur » et « MONSIEUR » soient comptés ensemble.

Ce nettoyage ne règle pas tout : des noms restent tronqués (Reine, Louis, Herman) ou sont des variantes d'un même nom (Johnson, M. Johnson). Nous le signalerons dans la conclusion.

In [20]:
# Entrées qui ne sont pas des personnes (raisons : voir le tableau du Markdown)
exclus = {
    "ROSSEL", "AGENCE ROSSEL",
    "ECRIRE AG", "TÉL", "F. TÉL", "EC AG",
    "BONNE", "LUNDI", "FAIRE", "TEMPS", "JARDIN", "OFFRE", "LIBRE", "TEINT",
    "BRUGMANN", "LOUISE", "MADOU", "ANSPACH",
    "PRÉSIDENT", "MONSIEUR", "MADAME",

}

In [21]:
presence_propres = Counter()

for txt in txts:
    with open(os.path.join(txt_path, txt), "r", encoding="utf-8") as f:
        doc = nlp(f.read())

    # 1. Nettoyage : tiret retiré, majuscules
    personnes = {ent.text.strip("- ").upper() for ent in doc.ents if ent.label_ == "PER"}

    # 2. On écarte les noms trop courts, les sigles avec un point, et la liste d'exclusion
    personnes = {p for p in personnes if len(p) > 3 and "." not in p and p not in exclus}

    presence_propres.update(personnes)

print(len(presence_propres), "personnes retenues")

10856 personnes retenues


In [22]:
for rang, (nom, n) in enumerate(presence_propres.most_common(200), start=1):
    print(f"{rang:2}. {nom} : {n} journaux")

 1. ALBERT : 15 journaux
 2. REINE : 14 journaux
 3. JOHNSON : 13 journaux
 4. MARTIN : 12 journaux
 5. CRÉD : 12 journaux
 6. MOLIÈRE : 11 journaux
 7. MOZART : 11 journaux
 8. BEETHOVEN : 10 journaux
 9. ATEL : 10 journaux
10. JANSSENS : 10 journaux
11. CURR : 10 journaux
12. PARF : 9 journaux
13. JARD : 9 journaux
14. CULS : 9 journaux
15. SHAKESPEARE : 9 journaux
16. AG ROSSEL : 9 journaux
17. COURONNE : 9 journaux
18. BACH : 9 journaux
19. MAGN : 8 journaux
20. BELLIARD : 8 journaux
21. LOUIS XV : 8 journaux
22. ROSSE : 8 journaux
23. GREN : 8 journaux
24. LOUIS : 8 journaux
25. LÉOPOLD : 7 journaux
26. PRINCE ALBERT : 7 journaux
27. HERMAN : 7 journaux
28. BRUXEL : 7 journaux
29. ROOSEVELT : 7 journaux
30. GÉNÉRAL DE GAULLE : 7 journaux
31. FRANÇOISE : 7 journaux
32. SCHUMANN : 7 journaux
33. BRAHMS : 7 journaux
34. FRANÇOIS : 6 journaux
35. JEAN : 6 journaux
36. GRÉTRY : 6 journaux
37. MARTINE : 6 journaux
38. THOMAS : 6 journaux
39. ANDERL : 6 journaux
40. 4 CHB : 6 journaux
41